In [10]:
import os
import pandas as pd
import numpy as np
import sys
import re
import logging
from Modules.Loader_wrangler import *
from Modules.Transformations import *
from Modules.ToTensor import *
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt

In [11]:
# Configure basic logging
logging.basicConfig(level=logging.INFO, force=True, format='%(levelname)s: %(message)s')

# Loading, Reading & Transforming to Tensors

In [3]:
# Define data range

years_to_extract = list(range(2017,2018))


In [4]:
df = loader(output_file_name="merged_df2017.pkl", chunksize=100000, sample_size=100000, survey_years=years_to_extract)

SurveyYear = ['2017'] not found in chunk 54. Continuing
Merged chunks saved to pickle!


In [12]:
#TODO get better file paths

nts_df = pd.read_pickle("/home/trapfishscott/Cambridge24.25/D200_ML_econ/ProblemSets/Project/data/merged_df2017.pkl")

### Exploring correlations with some of the target variables

In [8]:
ro_30_distance = return_correlated_columns(nts_df, ro=0.3, outcome_col="TripDisExSW")

Number of useful columns found: 9


In [9]:
# For TripPurpouse, default parameter for outcome col

ro_30_purpouse = return_correlated_columns(nts_df, ro=0.3)

Number of useful columns found: 9


In [16]:
print("Some useful columns for distance: ")
for col in ro_30_distance:
    print(col)
print("")
print("Some useful columns for Purpouse: ")
for col in ro_30_purpouse:
    print(col)

Some useful columns for distance: 
TripDisIncSW
JD
JTTXSC
TripTravTime
JOTXSC
TripTotalTime
TripTravTime_B01ID
TripTotalTime_B01ID
TripDisIncSW_B01ID

Some useful columns for Purpouse: 
ParkWk_B01ID
OftHome_B01ID
WkLift_B01ID
IndWkGOR_B02ID
EcoStat_B03ID
EcoStat_B02ID
PossHmN_B01ID
W5xHH
PDrivSt_B01ID


### Defining relevant variables

In [13]:
#TODO Make this somehow neater

#temporal_vars = ["TWSMonth", "TravelYear", "TravelWeekDay_B01ID"]
#individual_vars =["PSUGOR_B02ID", "IndIncome2002_B02ID", "HHoldNumChildren", "DVLALengthBand_B01ID"]

numerical_outcome_vars = ["TripStart", "TripEnd", "TripDisExSW"]
categorical_outcome_vars = ["TripPurpose_B01ID", "IsTrip"]

extra_vars = ["IndividualID_x", "JourSeq"]
features_one_hot = ["PSUGOR_B02ID"]


features_numerical = ["TravelYear", "ParkWk_B01ID", "OftHome_B01ID", "WkLift_B01ID", 'IndWkGOR_B02ID',
                     "EcoStat_B02ID", "PossHmN_B01ID", 'PDrivSt_B01ID',
                     "HHoldNumChildren", "IndIncome2002_B02ID", "DVLALengthBand_B01ID", 
                     'Age_B01ID', 
                     "DTJbLong_B01ID",
                    "DTJbMonth_B01ID",  
                    "EducN_B01ID",	
                    "HHoldEmploy_B01ID",	
                    "HHoldNumPeople",
                    "HHoldStruct_B02ID",	
                    "HRPWorkStat_B02ID",	
                    "WkMode_B01ID",	   
                    "WkPlace_B01ID",
                    "VehAnMileage"]
features_cyclical = ["TWSMonth", "TravelWeekDay_B01ID"]

features = features_one_hot + features_numerical + features_cyclical
outcomes = numerical_outcome_vars + categorical_outcome_vars

In [14]:
# TODO Missing mapping and missing features needs to be done when I include more data

'''
for feature in features:
    if feature not in df.columns:
        print(f"{k}: {feature}")

        # Appending feature with na
        df[feature] = 0
'''

'''
ts_df_full = pd.concat([ts_df2017, ts_df2018, ts_df2019, ts_df2020, ts_df2021])

ts_df_full = pd.concat([ts_df2017, ts_df2018, ts_df2019, ts_df2020, ts_df2021])

ts_df = ts_df2017.copy()'
'''

ts_df = nts_df[features + outcomes + extra_vars]

# Apply cyclical encoding to cyclical column, assuming the same categories that appear in 2017 appear elsewhere

standard_mms.fit_transform(ts_df[features_numerical])
ohe.fit_transform(ts_df[features_one_hot])

# Careful not to run twice

for col in features_one_hot:
    ts_df.loc[:,col] = ts_df.loc[:,col].astype(int)

ts_df.loc[:, "TravelWeekDay_B01ID"] = ts_df.loc[:, "TravelWeekDay_B01ID"].astype(int)

### Showcase: At random selects an individuals and shows the transformation process and the final features w/ indices

In [7]:

target_cols = prepare_data_for_LSTM(long_df=ts_df, 
                                    features=features,
                                    outcomes=outcomes,
                                    categorical_outcome_vars=categorical_outcome_vars,
                                    features_numerical=features_numerical,
                                    features_cyclical=features_cyclical,
                                    features_one_hot=features_one_hot,
                                    extra_vars=extra_vars,
                                    cyclical_encoder=apply_cyclical_encoding,
                                    custom_numerical_scaler=custom_numerical_scaler,
                                    log_transformer=log_transformer,
                                    debug=True,
                                    max_journey_seq=10, 
                                    seq_length = 7)

0: TripStart_1
1: TripEnd_1
2: TripStart_2
3: TripEnd_2
4: TripStart_3
5: TripEnd_3
6: TripStart_4
7: TripEnd_4
8: TripStart_5
9: TripEnd_5
10: TripStart_6
11: TripEnd_6
12: TripStart_7
13: TripEnd_7
14: TripStart_8
15: TripEnd_8
16: TripStart_9
17: TripEnd_9
18: TripStart_10
19: TripEnd_10
20: TripDisExSW_1
21: TripDisExSW_2
22: TripDisExSW_3
23: TripDisExSW_4
24: TripDisExSW_5
25: TripDisExSW_6
26: TripDisExSW_7
27: TripDisExSW_8
28: TripDisExSW_9
29: TripDisExSW_10
30: TripPurpose_B01ID_1
31: TripPurpose_B01ID_2
32: TripPurpose_B01ID_3
33: TripPurpose_B01ID_4
34: TripPurpose_B01ID_5
35: TripPurpose_B01ID_6
36: TripPurpose_B01ID_7
37: TripPurpose_B01ID_8
38: TripPurpose_B01ID_9
39: TripPurpose_B01ID_10
40: IsTrip_1
41: IsTrip_2
42: IsTrip_3
43: IsTrip_4
44: IsTrip_5
45: IsTrip_6
46: IsTrip_7
47: IsTrip_8
48: IsTrip_9
49: IsTrip_10
50: TravelYear
51: ParkWk_B01ID
52: OftHome_B01ID
53: WkLift_B01ID
54: IndWkGOR_B02ID
55: EcoStat_B02ID
56: PossHmN_B01ID
57: PDrivSt_B01ID
58: HHoldNumChi

,TripStart_1,TripEnd_1,TripStart_2,TripEnd_2,TripStart_3,TripEnd_3,TripStart_4,TripEnd_4,TripStart_5,TripEnd_5,...,TravelWeekDay_B01ID_sin,PSUGOR_B02ID_1.0,PSUGOR_B02ID_2.0,PSUGOR_B02ID_3.0,PSUGOR_B02ID_4.0,PSUGOR_B02ID_5.0,PSUGOR_B02ID_6.0,PSUGOR_B02ID_7.0,PSUGOR_B02ID_8.0,PSUGOR_B02ID_9.0
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.818315e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.749279e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
9,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.338837e-01,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


,TripStart_1,TripEnd_1,TripStart_2,TripEnd_2,TripStart_3,TripEnd_3,TripStart_4,TripEnd_4,TripStart_5,TripEnd_5,...,TripDisExSW_1,TripDisExSW_2,TripDisExSW_3,TripDisExSW_4,TripDisExSW_5,TripDisExSW_6,TripDisExSW_7,TripDisExSW_8,TripDisExSW_9,TripDisExSW_10
7,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.890372,2.890372,0.000000,0.000000,0.000000,0.000000,0,0,0,0
8,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.890372,2.890372,0.000000,0.000000,0.000000,0.000000,0,0,0,0
9,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.890372,2.890372,0.000000,0.000000,0.000000,0.000000,0,0,0,0
10,0.329861,0.347222,0.687500,0.708333,0.802083,0.805556,0.833333,0.837500,0.854167,0.859722,...,2.890372,2.890372,1.609438,1.609438,1.098612,1.098612,0,0,0,0
11,0.329861,0.347222,0.687500,0.708333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.890372,2.890372,0.000000,0.000000,0.000000,0.000000,0,0,0,0
12,0.416667,0.427083,0.479167,0.486111,0.500000,0.510417,0.697917,0.701389,0.708333,0.711806,...,1.252763,1.791759,1.386294,0.405465,0.405465,0.000000,0,0,0,0
13,0.546528,0.551389,0.597222,0.602083,0.625000,0.631944,0.666667,0.673611,0.000000,0.000000,...,1.609438,1.609438,2.397895,2.079442,0.000000,0.000000,0,0,0,0


,TripPurpose_B01ID_1,TripPurpose_B01ID_2,TripPurpose_B01ID_3,TripPurpose_B01ID_4,TripPurpose_B01ID_5,TripPurpose_B01ID_6,TripPurpose_B01ID_7,TripPurpose_B01ID_8,TripPurpose_B01ID_9,TripPurpose_B01ID_10,IsTrip_1,IsTrip_2,IsTrip_3,IsTrip_4,IsTrip_5,IsTrip_6,IsTrip_7,IsTrip_8,IsTrip_9,IsTrip_10
7,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0
8,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0
9,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0
10,1.0,1.0,16.0,16.0,5.0,5.0,0,0,0,0,1.0,1.0,1.0,1.0,1.0,1.0,0,0,0,0
11,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0,1.0,1.0,0.0,0.0,0.0,0.0,0,0,0,0
12,9.0,5.0,5.0,5.0,5.0,0.0,0,0,0,0,1.0,1.0,1.0,1.0,1.0,0.0,0,0,0,0
13,16.0,16.0,16.0,16.0,0.0,0.0,0,0,0,0,1.0,1.0,1.0,1.0,0.0,0.0,0,0,0,0


### Creating Tensors for Neural Network

In [5]:
X, y_cont, y_cat = prepare_data_for_LSTM(long_df=ts_df, 
                                            features=features,
                                            outcomes=outcomes,
                                            categorical_outcome_vars=categorical_outcome_vars,
                                            features_numerical=features_numerical,
                                            features_cyclical=features_cyclical,
                                            features_one_hot=features_one_hot,
                                            extra_vars=extra_vars,
                                            cyclical_encoder=apply_cyclical_encoding,
                                            custom_numerical_scaler=custom_numerical_scaler,
                                            log_transformer=log_transformer,
                                            debug=False,
                                            transform_to_wide=True, 
                                            transform_to_tensor=True,
                                            max_journey_seq=10, 
                                            seq_length = 7)

# Save tensors #TODO BETTER FILE PATH
with open("/home/trapfishscott/Cambridge24.25/D200_ML_econ/ProblemSets/Project/tensors/tensors.pkl", "wb") as f:
    pickle.dump((X, y_cont, y_cat), f)

(14, 85), (7, 30), (7, 20)
Individual 1 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 2 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 3 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 4 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 5 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 6 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 7 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 8 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 9 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 10 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 11 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 12 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 13 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 14 out of 6838 Complete!    (14, 85), (7, 30), (7, 20)
Individual 15 out of 6838 Complete!    (14, 85), (7, 30), 

### Importing Model

In [17]:
# Configure basic logging
logging.basicConfig(level=logging.INFO, force=True, format='%(levelname)s: %(message)s')

with open("/home/trapfishscott/Cambridge24.25/D200_ML_econ/ProblemSets/Project/Models/TravNet.pkl", "rb") as f:
    TravNet = pickle.load(f)

EOFError: Ran out of input